In [1]:
from selenium import webdriver
import time

driver = webdriver.Chrome()

driver.get("https://hrms.cgv.vn/")

time.sleep(5)

print("Title:", driver.title)
print("URL:", driver.current_url)

print(driver.find_element("tag name", "body").text[:10000])

driver.quit()

Title: CGV Cinemas Viet Nam
URL: https://hrms.cgv.vn/
Lịch chiếu CGV Cinemas Viet Nam
CGV Trường Sơn (CGV CT Plaza)
Tầng 10, Tòa nhà Vietjet Plaza, 60A Trường Sơn, P.2, Q. Tân Bình, TP.HCM, Việt Nam
25/09
HÔM NAY
26/09
THỨ BẢY
27/09
CHỦ NHẬT
28/09
THỨ HAI
29/09
THỨ BA
30/09
THỨ TƯ
01/10
THỨ NĂM
Khu vực:
                Hồ Chí Minh (20)
Hà Nội (23)
Đà Nẵng (2)
Hải Phòng (2)
Quảng Ninh (4)
Bà Rịa-Vũng Tàu (1)
Cần Thơ (3)
Đắk Lắk (1)
Bình Dương (3)
Đồng Nai (2)
Bình Định (1)
Yên Bái (1)
Trà Vinh (1)
Vĩnh Long (1)
Kiên Giang (1)
Hậu Giang (1)
Hà Tĩnh (1)
Phú Yên (1)
Khánh Hòa (1)
Thái Nguyên (1)
Sơn La (1)
Tây Ninh (1)
Lạng Sơn (1)
Tiền Giang (1)
Phú Thọ (1)
Quảng Ngãi (1)
Nghệ An (1)
Đồng Tháp (1)
Kon Tum (1)
Sóc Trăng (1)
Bạc Liêu (0)
             CGV TRƯỜNG SƠN (CGV CT PLAZA)
CGV PANDORA CITY
CGV AEON TÂN PHÚ
CGV LIBERTY CITYPOINT
CGV VINCOM THỦ ĐỨC
CGV VINCOM GÒ VẤP
CGV VINCOM ĐỒNG KHỞI
CGV PARKSON ĐỒNG KHỞI
CGV SƯ VẠN HẠNH
CGV SATRA CỦ CHI
CGV GIGA MALL THỦ ĐỨC
CGV LÝ CHÍNH THẮNG
CGV 

In [3]:
from selenium import webdriver
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome()

url = "https://www.cgv.vn/default/cinox/site/cgv-hoang-van-thu"

driver.get(url)
time.sleep(5)

soup = BeautifulSoup(driver.page_source, "html.parser")

print("TITLE:", driver.title)

# =========================
# LẤY NGÀY
# =========================

dates = soup.select("li.day.cgv-onlyone")

print("\n===== NGÀY =====")

for d in dates:
    print(
        d.get("id"),
        d.get_text(" ", strip=True)
    )


# =========================
# LẤY PHIM
# =========================

films = soup.select("div.film-list")

print("\n===== SỐ PHIM =====")
print("Có", len(films), "phim")


# =========================
# LẤY THÔNG TIN TỪNG PHIM
# =========================

for film in films:

    film_text = film.get_text(" ", strip=True)

    print("\n------------------------")
    print(film_text)

driver.quit()

TITLE: Site

===== NGÀY =====
cgv20260925 09 Fri 25
cgv20260926 09 Sat 26
cgv20260927 09 Sun 27
cgv20260928 09 Mon 28
cgv20260929 09 Tue 29
cgv20260930 09 Wed 30
cgv20261001 10 Thu 01
cgv20261002 10 Fri 02
cgv20261003 10 Sat 03
cgv20261004 10 Sun 04
cgv20261005 10 Mon 05
cgv20261006 10 Tue 06
cgv20261007 10 Wed 07
cgv20261008 10 Thu 08
cgv20261009 10 Fri 09
cgv20261010 10 Sat 10
cgv20261011 10 Sun 11
cgv20261012 10 Mon 12
cgv20261013 10 Tue 13
cgv20261014 10 Wed 14
cgv20261015 10 Thu 15
cgv20261016 10 Fri 16
cgv20261017 10 Sat 17
cgv20261018 10 Sun 18
cgv20261019 10 Mon 19
cgv20261020 10 Tue 20
cgv20261021 10 Wed 21
cgv20261022 10 Thu 22
cgv20261023 10 Fri 23
cgv20261024 10 Sat 24

===== SỐ PHIM =====
Có 3 phim

------------------------
TRẠI BUÔN NGƯỜI T18 2D Phụ Đề Anh 20:50 22:30 4DX2D Phụ Đề Anh | Rạp 4DX 23:20 2D Phụ Đề Anh | Rạp Lamour 22:50 2D Phụ Đề Anh 21:50

------------------------
VÙNG ĐẤT QUỶ DỮ T18 2D Phụ Đề Việt 21:00 23:30

------------------------
LÊN HƯƠNG T16 2D Phụ Đ

In [4]:
from selenium import webdriver
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

# =====================================================
# 1. KHỞI TẠO SELENIUM
# =====================================================

driver = webdriver.Chrome()

# Trang CGV
url = "https://www.cgv.vn/default/cinox/site/cgv-hoang-van-thu"

driver.get(url)
time.sleep(5)


# =====================================================
# 2. HÀM LẤY DỮ LIỆU TỪ 1 TRANG RẠP
# =====================================================

def scrape_cinema(driver, cinema_name, cinema_url):

    print("\n======================================")
    print("ĐANG CÀO:", cinema_name)
    print("======================================")

    driver.get(cinema_url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    data = []

    # -------------------------------------------------
    # Lấy danh sách ngày
    # -------------------------------------------------

    dates = soup.select("li.day.cgv-onlyone")

    print("Số ngày:", len(dates))

    # -------------------------------------------------
    # Duyệt từng ngày
    # -------------------------------------------------

    for date_element in dates:

        date_id = date_element.get("id")

        # Ví dụ:
        # cgv20260925
        # cgv20260926

        if not date_id:
            continue

        match = re.search(r"(\d{8})", date_id)

        if not match:
            continue

        date_str = match.group(1)

        show_date = (
            date_str[6:8] + "/" +
            date_str[4:6] + "/" +
            date_str[0:4]
        )

        print("\nNgày:", show_date)

        # -------------------------------------------------
        # Click ngày để tải suất chiếu tương ứng
        # -------------------------------------------------

        try:

            # Dùng JavaScript click
            driver.execute_script(
                "arguments[0].click();",
                driver.find_element(
                    "id",
                    date_id
                )
            )

            time.sleep(2)

        except Exception as e:

            print("Không click được ngày:", date_id)
            continue


        # Lấy HTML mới sau khi click
        soup = BeautifulSoup(
            driver.page_source,
            "html.parser"
        )


        # -------------------------------------------------
        # Lấy danh sách phim
        # -------------------------------------------------

        films = soup.select("div.film-list")

        print("Số phim:", len(films))


        # -------------------------------------------------
        # Duyệt từng phim
        # -------------------------------------------------

        for film in films:

            # =============================================
            # TÊN PHIM
            # =============================================

            film_name = ""

            # Tìm trong film-left
            film_left = film.select_one(".film-left")

            if film_left:

                # thử lấy alt của ảnh
                img = film_left.select_one("img")

                if img:

                    film_name = img.get("alt", "").strip()


                # nếu không có alt thì lấy text
                if not film_name:

                    film_name = film_left.get_text(
                        " ",
                        strip=True
                    )


            # Nếu vẫn không có tên
            if not film_name:

                continue


            # =============================================
            # LẤY CÁC ĐỊNH DẠNG + SUẤT CHIẾU
            # =============================================

            film_right = film.select_one(".film-right")

            if not film_right:
                continue


            # Các thẻ strong:
            #
            # 2D
            # 4DX
            # LAMOUR
            # ...

            formats = film_right.select(
                "strong.film-screen"
            )


            for format_element in formats:

                format_name = format_element.get_text(
                    " ",
                    strip=True
                )

                # div ngay sau strong
                showtime_div = format_element.find_next_sibling(
                    "div",
                    class_="film-showtimes"
                )

                if not showtime_div:
                    continue


                # Các nút giờ
                time_elements = showtime_div.select(
                    "a, button, li"
                )


                for time_element in time_elements:

                    show_time = time_element.get_text(
                        " ",
                        strip=True
                    )

                    # Chỉ lấy dạng HH:MM
                    match_time = re.search(
                        r"\b\d{1,2}:\d{2}\b",
                        show_time
                    )

                    if not match_time:
                        continue

                    show_time = match_time.group(0)


                    # =================================
                    # LƯU RECORD
                    # =================================

                    data.append({

                        "MovieName": film_name,

                        "Cinema": cinema_name,

                        "ShowDate": show_date,

                        "Format": format_name,

                        "ShowTime": show_time

                    })


    return data


# =====================================================
# 3. LẤY DANH SÁCH RẠP
# =====================================================

print("\nĐANG LẤY DANH SÁCH RẠP...")

# Quay về trang rạp
driver.get(
    "https://www.cgv.vn/default/cinox/site/cgv-hoang-van-thu"
)

time.sleep(3)

soup = BeautifulSoup(
    driver.page_source,
    "html.parser"
)


cinemas = soup.select(
    "li.cgv_city_1 span[onclick^='site(']"
)


print("Tìm thấy:", len(cinemas), "rạp")


cinema_list = []


for cinema in cinemas:

    cinema_name = cinema.get_text(
        " ",
        strip=True
    )

    onclick = cinema.get("onclick", "")

    # Lấy URL trong:
    # site('https://...')
    
    match = re.search(
        r"site\(['\"](.*?)['\"]",
        onclick
    )

    if not match:
        continue

    cinema_url = match.group(1)

    cinema_list.append(
        (cinema_name, cinema_url)
    )


# =====================================================
# 4. CHỈ CÀO MỘT SỐ RẠP ĐỂ TEST
# =====================================================

# Khi test:
# lấy 5 rạp trước

test_cinemas = cinema_list[:5]


print("\nSố rạp sẽ cào:", len(test_cinemas))


# =====================================================
# 5. CÀO DỮ LIỆU
# =====================================================

all_data = []


for cinema_name, cinema_url in test_cinemas:

    try:

        data = scrape_cinema(
            driver,
            cinema_name,
            cinema_url
        )

        all_data.extend(data)

        print(
            "→ Thu được:",
            len(data),
            "records"
        )

    except Exception as e:

        print(
            "LỖI:",
            cinema_name,
            e
        )


# =====================================================
# 6. ĐÓNG TRÌNH DUYỆT
# =====================================================

driver.quit()


# =====================================================
# 7. TẠO DATAFRAME
# =====================================================

df = pd.DataFrame(all_data)


# Xóa dòng trùng
df = df.drop_duplicates()


# Sắp xếp
df = df.sort_values(
    by=[
        "Cinema",
        "ShowDate",
        "MovieName",
        "ShowTime"
    ]
)


# =====================================================
# 8. HIỂN THỊ KẾT QUẢ
# =====================================================

print("\n======================================")
print("KẾT QUẢ")
print("======================================")

print("Số dòng:", len(df))

print("\n5 dòng đầu:")
print(df.head())


print("\nSố rạp:")
print(df["Cinema"].nunique())

print("\nSố phim:")
print(df["MovieName"].nunique())

print("\nSố ngày:")
print(df["ShowDate"].nunique())


# =====================================================
# 9. LƯU CSV
# =====================================================

df.to_csv(
    "cgv_showtimes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu: cgv_showtimes.csv")


ĐANG LẤY DANH SÁCH RẠP...
Tìm thấy: 21 rạp

Số rạp sẽ cào: 5

ĐANG CÀO: CGV Hùng Vương Plaza
Số ngày: 30

Ngày: 25/09/2026
Số phim: 5

Ngày: 26/09/2026
Số phim: 9

Ngày: 27/09/2026
Số phim: 8

Ngày: 28/09/2026
Số phim: 8

Ngày: 29/09/2026
Số phim: 8

Ngày: 30/09/2026
Số phim: 8

Ngày: 01/10/2026
Số phim: 8

Ngày: 02/10/2026
Số phim: 2

Ngày: 03/10/2026
Số phim: 2

Ngày: 04/10/2026
Số phim: 2

Ngày: 05/10/2026
Số phim: 0

Ngày: 06/10/2026
Số phim: 0

Ngày: 07/10/2026
Số phim: 0

Ngày: 08/10/2026
Số phim: 0

Ngày: 09/10/2026
Số phim: 0

Ngày: 10/10/2026
Số phim: 0

Ngày: 11/10/2026
Số phim: 0

Ngày: 12/10/2026
Số phim: 0

Ngày: 13/10/2026
Số phim: 0

Ngày: 14/10/2026
Số phim: 0

Ngày: 15/10/2026
Số phim: 0

Ngày: 16/10/2026
Số phim: 0

Ngày: 17/10/2026
Số phim: 0

Ngày: 18/10/2026
Số phim: 0

Ngày: 19/10/2026
Số phim: 0

Ngày: 20/10/2026
Số phim: 0

Ngày: 21/10/2026
Số phim: 0

Ngày: 22/10/2026
Số phim: 0

Ngày: 23/10/2026
Số phim: 0

Ngày: 24/10/2026
Số phim: 0
→ Thu được: 594 records


In [9]:
df = pd.read_csv(r"D:\Project_Group3\cgv_showtimes.csv")

df.head(10)

,MovieName,Cinema,ShowDate,Format,ShowTime
0,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,10:30
1,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,16:00
2,AVENGERS: HỒI KẾT - PHIÊN BẢN ĐẶC BIỆT (CHIẾU ...,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt | Rạp Lamour,21:30
3,CHIIKAWA: BÍ MẬT ĐẢO NGƯỜI CÁ,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh & Việt,18:00
4,HÒN ĐẢO QUÊN LÃNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,19:40
5,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,12:50
6,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Việt,14:40
7,LAPUTA: LÂU ĐÀI TRÊN KHÔNG,CGV Crescent Mall,01/10/2026,2D Lồng Tiếng Việt,17:10
8,LÊN HƯƠNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh,09:35
9,LÊN HƯƠNG,CGV Crescent Mall,01/10/2026,2D Phụ Đề Anh,10:20


In [8]:
len(df)

1172